## environment setup and library imports
professional setup for data analysis, visualization, and machine learning experiments.

In [12]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix, classification_report
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
import pickle

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)


## dataset loading and initialization
loading all datasets required for the integrated football analytics pipeline.

In [13]:
games = pd.read_csv(r'd:\Projects\WC26Analytics\games.csv', low_memory=False)
club_games = pd.read_csv(r'd:\Projects\WC26Analytics\club_games.csv')
game_lineups = pd.read_csv(r'd:\Projects\WC26Analytics\game_lineups.csv')
players = pd.read_csv(r'd:\Projects\WC26Analytics\players.csv')
appearances = pd.read_csv(r'd:\Projects\WC26Analytics\appearances.csv')
player_valuations = pd.read_csv(r'd:\Projects\WC26Analytics\player_valuations.csv')
transfers = pd.read_csv(r'd:\Projects\WC26Analytics\transfers.csv')
clubs = pd.read_csv(r'd:\Projects\WC26Analytics\clubs.csv')
competitions = pd.read_csv(r'd:\Projects\WC26Analytics\competitions.csv')
countries = pd.read_csv(r'd:\Projects\WC26Analytics\countries.csv')
national_teams = pd.read_csv(r'd:\Projects\WC26Analytics\national_teams.csv')
game_events = pd.read_csv(r'd:\Projects\WC26Analytics\game_events.csv')

for df in [games, game_lineups, player_valuations, transfers, appearances]:
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')

## data preparation
cleaning and preparing the merged dataset for feature engineering and modeling.

In [14]:
match_data = club_games.merge(games[['game_id', 'date', 'season', 'competition_id']],
                               on='game_id', how='left')
match_data = match_data.merge(competitions[['competition_id', 'type']],
                               left_on='competition_id', right_on='competition_id', how='left')

match_data = match_data.dropna(subset=['date', 'own_goals', 'opponent_goals'])
match_data['year'] = match_data['date'].dt.year
match_data = match_data[match_data['year'] >= 2014].copy().sort_values('date').reset_index(drop=True)

# create 3-class target
goal_diff = match_data['own_goals'] - match_data['opponent_goals']
match_data['target_class'] = pd.cut(goal_diff, bins=[-np.inf, -0.1, 0.1, np.inf],
                                     labels=['Loss', 'Draw', 'Win'])
match_data['target_encoded'] = match_data['target_class'].map({'Loss': 0, 'Draw': 1, 'Win': 2})

## feature engineering
constructing performance, market value, transfer, and event-based features.

In [15]:
target_dist = match_data['target_class'].value_counts()
print(f"\nClass distribution:")
print(f"  Wins:  {target_dist['Win']:7d} ({target_dist['Win']/len(match_data):.1%})")
print(f"  Draws: {target_dist['Draw']:7d} ({target_dist['Draw']/len(match_data):.1%})")
print(f"  Loss:  {target_dist['Loss']:7d} ({target_dist['Loss']/len(match_data):.1%})")


Class distribution:
  Wins:    61868 (39.2%)
  Draws:   34102 (21.6%)
  Loss:    61868 (39.2%)


In [16]:
# Time-based split
split_date = match_data['date'].quantile(0.8)
train_data = match_data[match_data['date'] < split_date].copy()
test_data = match_data[match_data['date'] >= split_date].copy()

# Optimized time-bounded feature engineering using groupby + merge
def create_time_bounded_features_fast(data_for_features, window_days=365):
    features_list = []
    
    # Pre-sort by date for efficient filtering
    match_data_sorted = match_data.sort_values('date').reset_index(drop=True)
    
    for idx, match_row in data_for_features.iterrows():
        match_date = match_row['date']
        club_id = match_row['club_id']
        opp_id = match_row['opponent_id']
        
        window_start = match_date - pd.Timedelta(days=window_days)
        
        # Team history
        team_hist = match_data_sorted[(match_data_sorted['club_id'] == club_id) &
                                       (match_data_sorted['date'] < match_date) &
                                       (match_data_sorted['date'] > window_start)]
        
        if len(team_hist) > 0:
            team_encoded = (team_hist['target_encoded'] == 2).astype(int)
            team_win_rate = team_encoded.mean()
            team_gf = team_hist['own_goals'].mean()
            team_ga = team_hist['opponent_goals'].mean()
            team_draw_rate = (team_hist['target_encoded'] == 1).mean()
        else:
            team_win_rate = team_gf = team_ga = team_draw_rate = 0.5
        
        # Opponent history
        opp_hist = match_data_sorted[(match_data_sorted['club_id'] == opp_id) &
                                      (match_data_sorted['date'] < match_date) &
                                      (match_data_sorted['date'] > window_start)]
        
        if len(opp_hist) > 0:
            opp_encoded = (opp_hist['target_encoded'] == 2).astype(int)
            opp_win_rate = opp_encoded.mean()
            opp_gf = opp_hist['own_goals'].mean()
            opp_ga = opp_hist['opponent_goals'].mean()
            opp_draw_rate = (opp_hist['target_encoded'] == 1).mean()
        else:
            opp_win_rate = opp_gf = opp_ga = opp_draw_rate = 0.5
        
        features_list.append({
            'team_win_rate': team_win_rate,
            'team_gf': team_gf,
            'team_ga': team_ga,
            'team_draw_rate': team_draw_rate,
            'opp_win_rate': opp_win_rate,
            'opp_gf': opp_gf,
            'opp_ga': opp_ga,
            'opp_draw_rate': opp_draw_rate,
        })
    
    return pd.DataFrame(features_list)

print("Computing train features...")
train_features = create_time_bounded_features_fast(train_data)
train_data = pd.concat([train_data.reset_index(drop=True), train_features], axis=1)

print("Computing test features...")
test_features = create_time_bounded_features_fast(test_data)
test_data = pd.concat([test_data.reset_index(drop=True), test_features], axis=1)

# Squad market values
squad_vals = player_valuations.groupby('current_club_id')['market_value_in_eur'].agg(['mean', 'sum']).reset_index()
squad_vals.columns = ['club_id', 'avg_val', 'total_val']

train_data = train_data.merge(squad_vals, on='club_id', how='left')
squad_vals_opp = squad_vals.copy()
squad_vals_opp.columns = ['opponent_id', 'avg_val_o', 'total_val_o']
train_data = train_data.merge(squad_vals_opp, on='opponent_id', how='left')

test_data = test_data.merge(squad_vals, on='club_id', how='left')
test_data = test_data.merge(squad_vals_opp, on='opponent_id', how='left')

train_data['value_diff'] = train_data['total_val'].fillna(0) - train_data['total_val_o'].fillna(0)
train_data['player_qual_diff'] = train_data['avg_val'].fillna(0) - train_data['avg_val_o'].fillna(0)

test_data['value_diff'] = test_data['total_val'].fillna(0) - test_data['total_val_o'].fillna(0)
test_data['player_qual_diff'] = test_data['avg_val'].fillna(0) - test_data['avg_val_o'].fillna(0)

# Transfers activity
trans_dict = transfers.groupby('to_club_id').size().to_dict()
train_data['transfers_team'] = train_data['club_id'].map(trans_dict).fillna(0)
train_data['transfers_opp'] = train_data['opponent_id'].map(trans_dict).fillna(0)

test_data['transfers_team'] = test_data['club_id'].map(trans_dict).fillna(0)
test_data['transfers_opp'] = test_data['opponent_id'].map(trans_dict).fillna(0)

# Game events - vectorized
events_dict = game_events.groupby(['game_id', 'club_id']).size().to_dict()
train_data['events_team'] = train_data[['game_id', 'club_id']].apply(
    lambda r: events_dict.get((r['game_id'], r['club_id']), 0), axis=1)
train_data['events_opp'] = train_data[['game_id', 'opponent_id']].apply(
    lambda r: events_dict.get((r['game_id'], r['opponent_id']), 0), axis=1)

test_data['events_team'] = test_data[['game_id', 'club_id']].apply(
    lambda r: events_dict.get((r['game_id'], r['club_id']), 0), axis=1)
test_data['events_opp'] = test_data[['game_id', 'opponent_id']].apply(
    lambda r: events_dict.get((r['game_id'], r['opponent_id']), 0), axis=1)

# Home feature and derived features
train_data['is_home'] = (train_data['hosting'] == 'Home').astype(int)
train_data['strength_diff'] = (train_data['team_gf'] - train_data['team_ga']) - (train_data['opp_gf'] - train_data['opp_ga'])

test_data['is_home'] = (test_data['hosting'] == 'Home').astype(int)
test_data['strength_diff'] = (test_data['team_gf'] - test_data['team_ga']) - (test_data['opp_gf'] - test_data['opp_ga'])

# Fill NaN values efficiently
numeric_cols_train = train_data.select_dtypes(include=[np.number]).columns
for col in numeric_cols_train:
    fill_val = train_data[col].median()
    train_data[col].fillna(fill_val, inplace=True)
    test_data[col].fillna(fill_val, inplace=True)

Computing train features...
Computing test features...


## statistical analysis
analyzing feature correlations and target relationships.

In [17]:
numeric_cols = train_data.select_dtypes(include=[np.number]).columns.tolist()
cols_for_corr = numeric_cols + ['target_encoded'] if 'target_encoded' not in numeric_cols else numeric_cols
corr_with_target = train_data[cols_for_corr].corr()['target_encoded'].sort_values(ascending=False)

print(f"\n  Top 15 correlations with outcome:")
for feat, corr in corr_with_target.head(15).items():
    if feat != 'target_encoded':
        print(f"    {feat:35s}: {corr:.4f}")


  Top 15 correlations with outcome:
    is_win                             : 0.9079
    own_goals                          : 0.5964
    opponent_position                  : 0.3230
    value_diff                         : 0.2967
    events_team                        : 0.2936
    player_qual_diff                   : 0.2817
    strength_diff                      : 0.2124
    total_val                          : 0.1654
    avg_val                            : 0.1597
    is_home                            : 0.1293
    team_win_rate                      : 0.1151
    team_gf                            : 0.1086
    opp_ga                             : 0.0813
    transfers_team                     : 0.0802


## visualization pipeline
generating visual analytics for interpretation and reporting.

In [18]:
# correlation heatmap
top_20 = list(corr_with_target[corr_with_target.index != 'target_encoded'].head(18).index) + ['target_encoded']
corr_mat = train_data[top_20].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, square=True)
ax.set_title('feature correlations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_01_correlations.png', dpi=300, bbox_inches='tight')
plt.close()

# distributions by class
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, col in enumerate(['team_win_rate', 'strength_diff', 'value_diff', 'player_qual_diff']):
    ax = axes[idx // 2, idx % 2]
    for cls_val, cls_name, color in [(0, 'Loss', '#e74c3c'), (1, 'Draw', '#f39c12'), (2, 'Win', '#2ecc71')]:
        ax.hist(train_data[train_data['target_encoded']==cls_val][col], bins=20, alpha=0.5, label=cls_name, color=color)
    ax.set_xlabel(col)
    ax.set_title(f'{col} by Outcome')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_02_distributions.png', dpi=300, bbox_inches='tight')
plt.close()

## model preparation
preparing features, scaling data, and splitting train/test sets.

In [19]:
exclude_cols = ['game_id', 'club_id', 'opponent_id', 'date', 'hosting', 'target_class',
                'target_encoded', 'type', 'own_goals', 'opponent_goals', 'competition_id',
                'season', 'goal_diff', 'total_goals', 'year']

X_cols = [c for c in train_data.columns if c not in exclude_cols and c in numeric_cols]

X_train = train_data[X_cols].values
y_train = train_data['target_encoded'].values

X_test = test_data[X_cols].values
y_test = test_data['target_encoded'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"  features: {len(X_cols)}")
print(f"  train: {X_train_scaled.shape} | test: {X_test_scaled.shape}")

  features: 23
  train: (126238, 23) | test: (31600, 23)


## model comparison and training
training multiple ensemble learning models and comparing performance.

In [20]:
models = {
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=7, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=7, random_state=42, verbosity=0),
}

results = []
best_model = None
best_name = None
best_acc = 0

for name, model in models.items():
    print(f"  training {name}...")
    model.fit(X_train_scaled, y_train)

    train_acc = model.score(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)

    results.append({
        'Model': name,
        'Train_Accuracy': train_acc,
        'Test_Accuracy': test_acc,
    })

    if test_acc > best_acc:
        best_acc = test_acc
        best_model = model
        best_name = name

comp = pd.DataFrame(results).sort_values('Test_Accuracy', ascending=False)
comp.to_csv(r'd:\Projects\WC26Analytics\model_comparison.csv', index=False)

print("\n  Model Results:")
print(comp.to_string(index=False))

# visualization
fig, ax = plt.subplots(figsize=(12, 6))
comp_sort = comp.sort_values('Test_Accuracy')
colors = ['#2ecc71' if m == best_name else '#3498db' for m in comp_sort['Model']]
ax.barh(comp_sort['Model'], comp_sort['Test_Accuracy'], color=colors)
ax.set_xlabel('Test Accuracy')
ax.set_title('Multi-Class Model Comparison', fontsize=14, fontweight='bold')
ax.set_xlim([0, 1])
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_03_model_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

  training Gradient Boosting...
  training Random Forest...
  training XGBoost...

  Model Results:
            Model  Train_Accuracy  Test_Accuracy
Gradient Boosting        0.898945       0.833070
    Random Forest        0.930251       0.826962
          XGBoost        0.951354       0.823576


## final evaluation and interpretation
evaluating the best-performing model using classification metrics and visualizations.

In [21]:
y_pred = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)

# Confusion matrix (3-class)
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
            xticklabels=['Loss', 'Draw', 'Win'], yticklabels=['Loss', 'Draw', 'Win'])
ax.set_title(f'Confusion Matrix - {best_name} (Accuracy: {best_acc:.4f})', fontsize=14, fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_04_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

# ROC curves for multi-class (One-vs-Rest) - Fixed
from sklearn.preprocessing import label_binarize
y_bin = label_binarize(y_test, classes=[0, 1, 2])

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c', '#f39c12', '#2ecc71']
class_names = ['Loss', 'Draw', 'Win']

for i, (color, class_name) in enumerate(zip(colors, class_names)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=3, label=f'{class_name} (AUC={roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves (One-vs-Rest)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_05_roc_curve.png', dpi=300, bbox_inches='tight')
plt.close()

# PR curves for multi-class (One-vs-Rest) - Fixed
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c', '#f39c12', '#2ecc71']
class_names = ['Loss', 'Draw', 'Win']

for i, (color, class_name) in enumerate(zip(colors, class_names)):
    prec, rec, _ = precision_recall_curve(y_bin[:, i], y_proba[:, i])
    pr_auc = auc(rec, prec)
    baseline = (y_test == i).mean()
    ax.plot(rec, prec, color=color, lw=3, label=f'{class_name} (AUC={pr_auc:.4f})')
    ax.axhline(baseline, color=color, linestyle=':', alpha=0.5)

ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curves (One-vs-Rest)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_06_pr_curve.png', dpi=300, bbox_inches='tight')
plt.close()

# Feature importance
feat_imp = pd.DataFrame({
    'feature': X_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

top_n = 20
top_imp = feat_imp.head(top_n)

fig, ax = plt.subplots(figsize=(12, 8))
colors_imp = plt.cm.viridis(np.linspace(0, 1, len(top_imp)))
ax.barh(range(len(top_imp)), top_imp['importance'].values, color=colors_imp)
ax.set_yticks(range(len(top_imp)))
ax.set_yticklabels(top_imp['feature'].values, fontsize=9)
ax.set_xlabel('Importance', fontsize=11)
ax.set_title(f'Top {top_n} Features - Multi-Class', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(r'd:\Projects\WC26Analytics\viz_07_feature_importances.png', dpi=300, bbox_inches='tight')
plt.close()

feat_imp.to_csv(r'd:\Projects\WC26Analytics\feature_importances.csv', index=False)

print(f"\n  top 20 features:")
for idx, row in feat_imp.head(20).iterrows():
    print(f"    {row['feature']:35s} | {row['importance']:.6f}")

# Save models
with open(r'd:\Projects\WC26Analytics\best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open(r'd:\Projects\WC26Analytics\scaler_final.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(r'd:\Projects\WC26Analytics\feature_columns_final.pkl', 'wb') as f:
    pickle.dump(X_cols, f)


  top 20 features:
    is_win                              | 0.781520
    events_opp                          | 0.055460
    opponent_position                   | 0.040301
    own_position                        | 0.018602
    events_team                         | 0.018218
    value_diff                          | 0.008509
    player_qual_diff                    | 0.007773
    is_home                             | 0.006983
    opp_win_rate                        | 0.005870
    transfers_team                      | 0.005335
    strength_diff                       | 0.004931
    team_draw_rate                      | 0.004745
    team_gf                             | 0.004617
    team_ga                             | 0.004529
    team_win_rate                       | 0.004464
    opp_gf                              | 0.004290
    opp_draw_rate                       | 0.003979
    opp_ga                              | 0.003961
    total_val                           | 0.003499
    transfe

## project summary
final overview of datasets, features, performance metrics, and generated outputs.

In [22]:
print("Project Summary")
print("="*80)
print(f"\nData Summary:")
print(f"  Train: {len(train_data)} matches ({train_data['date'].min().date()} to {train_data['date'].max().date()})")
print(f"  Test:  {len(test_data)} matches ({test_data['date'].min().date()} to {test_data['date'].max().date()})")
print(f"  Features: {len(X_cols)}")
print(f"\nBest Model: {best_name}")
print(f"  Train Accuracy: {comp.iloc[0]['Train_Accuracy']:.4f}")
print(f"  Test Accuracy: {best_acc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Loss', 'Draw', 'Win']))

Project Summary

Data Summary:
  Train: 126238 matches (2014-01-01 to 2024-11-01)
  Test:  31600 matches (2024-11-02 to 2026-05-07)
  Features: 23

Best Model: Gradient Boosting
  Train Accuracy: 0.8989
  Test Accuracy: 0.8331

Classification Report:
              precision    recall  f1-score   support

        Loss       0.76      0.82      0.79     12202
        Draw       0.65      0.57      0.61      7196
         Win       1.00      1.00      1.00     12202

    accuracy                           0.83     31600
   macro avg       0.81      0.80      0.80     31600
weighted avg       0.83      0.83      0.83     31600

